In [17]:
import pandas as pd
import networkx as nx

prereqs = pd.read_csv(
    "../data/processed/prerequisites_fall_2026.csv"
)

prereqs["course"] = (
    prereqs["subject"] + " " + prereqs["course_number"].astype(str)
)

prereqs["prerequisite"] = (
    prereqs["prereq_subject"] + " " + prereqs["prereq_number"].astype(str)
)

G = nx.DiGraph()

for _, row in prereqs.iterrows():
    G.add_edge(
        row["prerequisite"],
        row["course"]
    )

In [18]:
completed_courses = {
    "DATA C8",
    "COMPSCI 61A",
    "MATH 1A",
    "MATH 1B"
}

In [19]:
def get_prerequisites(course):
    if course not in G:
        return set()

    return set(G.predecessors(course))

In [20]:
def is_eligible(course, completed_courses):
    prereqs = get_prerequisites(course)

    return prereqs.issubset(completed_courses)

In [21]:
get_prerequisites("AEROENG 10")

{'COMPSCI 61A', 'ENGIN 7', 'MATH 51', 'MATH 52', 'MATH 53', 'PHYSICS 7A'}

In [22]:
def missing_prerequisites(course, completed_courses):
    return get_prerequisites(course) - completed_courses

In [23]:
missing_prerequisites("AEROENG 10", completed_courses)

{'ENGIN 7', 'MATH 51', 'MATH 52', 'MATH 53', 'PHYSICS 7A'}

In [24]:
is_eligible("AEROENG 10", completed_courses)

False

In [25]:
courses = pd.read_csv(
    "../data/processed/recommendable_courses_fall_2026.csv"
)

courses["course"] = (
    courses["subject"] + " " + courses["course_number"].astype(str)
)

courses["missing_prereqs"] = courses["course"].apply(
    lambda course: missing_prerequisites(course, completed_courses)
)

courses["eligible"] = courses["missing_prereqs"].apply(
    lambda missing: len(missing) == 0
)

In [26]:
courses["eligible"].value_counts()

eligible
True     1683
False     394
Name: count, dtype: int64

In [27]:
eligible_courses = courses[
    courses["eligible"]
].copy()

eligible_courses.shape

(1683, 11)

In [28]:
eligible_courses[
    ["course", "title", "requirements"]
].sample(20, random_state=42)

,course,title,requirements
1751,RHETOR 157A,Rhetoric of Modern Political Theory,NaN
937,HISTORY C196B,UCDC INTERNSHIP,C196A (must be taken concurrently).
1993,UGBA 195T,Topics in Entrepreneurship,NaN
53,AMERSTD 101AC,Examining U.S. Cultures in Time,NaN
240,CHEM 197,Field Study in Chemistry,Upper division standing and consent of instruc...
657,EPS 116,Structural Geology and Tectonics,50
1122,JEWISH 175E,History of Modern Israel: From the Emergence o...,NaN
803,FILM 188,Documentary and Non-fiction Media Production,NaN
2018,URDU 1A,Introductory Urdu,NaN
1694,PORTUG 102,Conversations in Portuguese,Portuguese 50 or equivalent


In [29]:
courses.loc[
    ~courses["eligible"],
    ["course", "title", "missing_prereqs"]
].head(20)

,course,title,missing_prereqs
1,AEROENG 10,Introduction to Aerospace Engineering Design,"{MATH 52, MATH 51, MATH 53, PHYSICS 7A, ENGIN 7}"
2,AEROENG 100,Aerospace Capstone,"{MECENG 132, MECENG 103, MECENG 106, MECENG 104}"
4,AEROENG C124,Materials for Extreme Environments,{ENGIN 40}
7,MECENG C162,Introduction to Flight Mechanics,"{MATH 52, PHYSICS 7A}"
66,ANTHRO 106,Primate Behavior,{BIOLOGY 32}
72,ANTHRO 127A,Bioarchaeology: Introduction to Skeletal Biolo...,{BIOLOGY 1B}
78,ANTHRO 150,Utopia: Art and Power in Modern Times,{ANTHRO 3}
90,ARABIC 100A,Advanced Arabic,"{ARABIC 20B, ARABIC 30}"
91,ARABIC 104A,Modern Arabic Prose,"{ARABIC 20B, ARABIC 30}"
92,ARABIC 1A,Elementary Arabic,"{ARABIC 1A, ARABIC 1B}"


### Eligibility Observations
- For the example student profile, 1,725 of 2,119 recommendable courses have no missing parsed course prerequisites.
- 394 courses are blocked by one or more explicit course prerequisites. 
- eligible=True only means that no parsed course prerequisites are missing; non-course requirements such as instructor consent, auditions, GPA thresholds, or standing restrictions are not yet enforced.

In [31]:
completed_courses = {
    "DATA C8",
    "COMPSCI 61A",
    "MATH 1A",
    "MATH 1B",
}

for course in [
    "DATA C102",
    "COMPSCI 189",
    "STAT 154",
    "DATA C100",
]:
    print(course)
    print("Prereqs:", get_prerequisites(course))
    print("Missing:", missing_prerequisites(course, completed_courses))
    print()

DATA C102
Prereqs: {'STAT 134', 'STAT 89A', 'EECS 126', 'COMPSCI C100', 'MATH 110', 'ELENG 64', 'PHYSICS 89', 'ELENG 66', 'MATH 54', 'INDENG 172', 'MATH 56', 'MATH 106', 'STAT C140'}
Missing: {'STAT 134', 'STAT 89A', 'EECS 126', 'COMPSCI C100', 'MATH 110', 'ELENG 64', 'PHYSICS 89', 'ELENG 66', 'MATH 54', 'INDENG 172', 'MATH 56', 'MATH 106', 'STAT C140'}

COMPSCI 189
Prereqs: {'MATH 54', 'MATH 53', 'COMPSCI 70'}
Missing: {'MATH 54', 'MATH 53', 'COMPSCI 70'}

STAT 154
Prereqs: {'STAT C140', 'COMPSCI C100'}
Missing: {'STAT C140', 'COMPSCI C100'}

DATA C100
Prereqs: {'STAT 20', 'DATA C8', 'DATA C100', 'COMPSCI 61A', 'DATA C88C', 'MATH 110', 'PHYSICS 89', 'ELENG 66', 'MATH 54', 'MATH 56', 'ENGIN 7'}
Missing: {'STAT 20', 'DATA C100', 'DATA C88C', 'MATH 110', 'PHYSICS 89', 'ELENG 66', 'MATH 54', 'MATH 56', 'ENGIN 7'}

